# Vietnamese ABSA Demo

This notebook demonstrates the Vietnamese Aspect-Based Sentiment Analysis system.

## Setup

In [ ]:
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

from src.preprocessing import create_preprocessor
from src.inference import create_pipeline
from src.utils import load_config
import json

## 1. Text Preprocessing Demo

In [ ]:
# Create preprocessor
preprocessor = create_preprocessor("../config/config.yaml")

# Example text
text = "Điện thoại này ko tốt lắm, màn hình xấu quá 😢"

# Preprocess
result = preprocessor.preprocess(text, return_masks=True)

print("Original text:", result['original_text'])
print("Processed text:", result['processed_text'])
print("Negation mask:", result['negation_mask'])
print("Intensity mask:", result['intensity_mask'])
print("Emojis:", result['emojis'])

## 2. Full ABSA Pipeline Demo

In [ ]:
# Note: This requires trained models
# If models are not available, this cell will raise an error

try:
    # Load pipeline
    pipeline = create_pipeline(
        stage1_model_path="../models/stage1_aspect_extraction",
        stage2_model_path="../models/stage2_sentiment",
        config_path="../config/config.yaml"
    )
    
    # Example texts
    texts = [
        "Điện thoại này màn hình đẹp nhưng pin yếu quá",
        "Máy chạy rất mượt, hiệu năng tốt",
        "Camera chụp ảnh đẹp, màu sắc sống động",
        "Thiết kế đẹp mắt, nhưng giá hơi cao"
    ]
    
    # Predict
    for text in texts:
        print("\n" + "="*80)
        print(f"Text: {text}")
        print("="*80)
        
        result = pipeline.predict(text)
        
        print(f"Overall Sentiment: {result['overall_sentiment']}")
        print("\nAspects:")
        for aspect in result['aspects']:
            print(f"  - {aspect['aspect']}: {aspect['sentiment']} (confidence: {aspect['confidence']:.2%})")

except Exception as e:
    print(f"Error: {e}")
    print("\nNote: This demo requires trained models.")
    print("Please train the models first using:")
    print("  python scripts/train_stage1.py")
    print("  python scripts/train_stage2.py")

## 3. Data Validation Demo

In [ ]:
# Example: Create and validate sample data
sample_data = [
    {
        "text": "Điện thoại này màn hình đẹp nhưng pin yếu quá",
        "aspects": [
            {"term": "màn hình", "span": [18, 27], "sentiment": "positive"},
            {"term": "pin", "span": [35, 38], "sentiment": "negative"}
        ],
        "overall_sentiment": "mixed"
    },
    {
        "text": "Máy chạy rất mượt, hiệu năng tốt",
        "aspects": [
            {"term": "hiệu năng", "span": [21, 30], "sentiment": "positive"}
        ],
        "overall_sentiment": "positive"
    }
]

# Display
print("Sample ABSA Data:")
print(json.dumps(sample_data, ensure_ascii=False, indent=2))

## 4. Lexicon Coverage Demo

In [ ]:
# Load configuration
config = load_config("../config/config.yaml")

# Display lexicon sizes
preprocessing_config = config.get('preprocessing', {})

print("Lexicon Sizes:")
print(f"  Negation words: {len(preprocessing_config.get('negation_words', []))}")
print(f"  Intensity words: {len(preprocessing_config.get('intensity_words', []))}")
print(f"  Contrast words: {len(preprocessing_config.get('contrast_words', []))}")
print(f"  Positive emojis: {len(preprocessing_config.get('positive_emojis', []))}")
print(f"  Negative emojis: {len(preprocessing_config.get('negative_emojis', []))}")

# Sample words
print("\nSample negation words:", preprocessing_config.get('negation_words', [])[:10])
print("Sample intensity words:", preprocessing_config.get('intensity_words', [])[:10])

## 5. Model Architecture Demo

In [ ]:
# Display model configurations
stage1_config = load_config("../config/training_stage1.yaml")
stage2_config = load_config("../config/training_stage2.yaml")

print("Stage 1 (Aspect Extraction):")
print(f"  Model: {stage1_config['model']['backbone']}")
print(f"  Labels: {stage1_config['model']['num_labels']} (B-ASP, I-ASP, O)")
print(f"  Epochs: {stage1_config['training']['num_train_epochs']}")
print(f"  Batch size: {stage1_config['training']['per_device_train_batch_size']}")
print(f"  Learning rate: {stage1_config['training']['learning_rate']}")

print("\nStage 2 (Sentiment Classification):")
print(f"  Model: {stage2_config['model']['backbone']}")
print(f"  Labels: {stage2_config['model']['num_labels']} (positive, negative, neutral, mixed)")
print(f"  Epochs: {stage2_config['training']['num_train_epochs']}")
print(f"  Use negation features: {stage2_config['model'].get('use_negation_features', False)}")
print(f"  Use intensity features: {stage2_config['model'].get('use_intensity_features', False)}")

## Conclusion

This notebook demonstrates the key components of the Vietnamese ABSA system:

1. **Preprocessing**: Vietnamese text normalization with lexicons
2. **ABSA Pipeline**: Two-stage aspect extraction and sentiment classification
3. **Data Validation**: Proper annotation format
4. **Lexicon Coverage**: Comprehensive Vietnamese sentiment lexicons
5. **Model Architecture**: PhoBERT-based dual-stage system

For production use, train the models on your domain-specific data following the training guide in the README.